In [1]:
import numpy as np
import pandas as pd
import mne 
import matplotlib.pyplot as plt
mne.set_log_level('error')
import warnings
warnings.simplefilter("ignore")
import seaborn as sns

In [2]:
df = pd.read_csv('EEGdata.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'EEGdata.csv'

In [ ]:
events_array = df['Validation']
df = df.drop(columns = ['Time','Battery','Counter'])
EEG_data = df.values.T * 1e-6  # Convert μV to Volts
ch_names = df.columns[0:8].tolist()
ch_types = ['eeg']*len(ch_names)

In [ ]:
df

In [ ]:
info = mne.create_info(ch_names = ch_names, ch_types = ch_types, sfreq = 250)
raw = mne.io.RawArray(EEG_data[0:8], info)
raw

In [ ]:
raw.describe()

In [ ]:
raw.info['dig']

In [ ]:
raw.info['ch_names']

In [ ]:
mapping = {
    'FZ': 'Fz',
    'CZ': 'Cz',
    'C3': 'C3',
    'C4': 'C4',
    'PZ': 'Pz',
    'PO7': 'PO7',
    'OZ': 'Oz',
    'PO8': 'PO8'
}

raw.rename_channels(mapping)


In [ ]:
picks = mne.pick_channels_regexp(raw.ch_names, regexp="['FZ', 'C3', 'CZ', 'C4', 'PZ', 'PO7', 'OZ', 'PO8']")
raw.plot(order=picks, n_channels=len(picks), scalings = 'auto');

In [ ]:
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

In [ ]:
raw.plot_sensors(show_names = True)
plt.show()

# Motor Imagery(MI)

Channel selection: C3, C4, Cz, Fz, accelerometer norm, gyroscope norm, and the validation signal.

In [ ]:
df.columns = df.columns.str.strip()

# Accelerometer norm
df['Acc_norm'] = np.sqrt(df['AccX']**2 + df['AccY']**2 + df['AccZ']**2)

# Gyroscope norm
df['Gyro_norm'] = np.sqrt(df['Gyro1']**2 + df['Gyro2']**2 + df['Gyro3']**2)


In [ ]:
# Step 3: Set thresholds
# -------------------------------
acc_threshold = df['Acc_norm'].quantile(0.95)
gyro_threshold = df['Gyro_norm'].quantile(0.95)


In [ ]:
# Step 4: Filter segments
# -------------------------------
high_acc = df[df['Acc_norm'] > acc_threshold]
high_gyro = df[df['Gyro_norm'] > gyro_threshold]
val_1 = df[df['Validation'] == 1]
val_0 = df[df['Validation'] == 0]


In [ ]:
# Step 5: Visualize EEG signals with high gyro
# -------------------------------
plt.figure(figsize=(14, 6))
for ch in ['FZ', 'C3', 'CZ', 'C4']:
    plt.plot(high_gyro[ch].values[:200], label=ch)
plt.title("EEG Signals during High Gyroscope Activity")
plt.xlabel("Sample Index")
plt.ylabel("EEG Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Visualize EEG signals with high accelerometer
# -------------------------------
plt.figure(figsize=(14, 6))
for ch in ['FZ', 'C3', 'CZ', 'C4']:
    plt.plot(high_acc[ch].values[:200], label=ch)
plt.title("EEG Signals during High Accelerometer Activity")
plt.xlabel("Sample Index")
plt.ylabel("EEG Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Compare Validation == 1 vs 0
# -------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

In [ ]:
# Validation = 1
for ch in ['FZ', 'C3', 'CZ', 'C4']:
    axes[0].plot(val_1[ch].values[:200], label=ch)
axes[0].set_title("EEG Signals when Validation = 1 (Task Active)")
axes[0].legend()
axes[0].grid(True)

# Validation = 0
for ch in ['FZ', 'C3', 'CZ', 'C4']:
    axes[1].plot(val_0[ch].values[:200], label=ch)
axes[1].set_title("EEG Signals when Validation = 0 (Rest)")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for selected EEG channels
eeg_channels = ['FZ', 'C3', 'CZ', 'C4']
print("Summary Stats:\n", df[eeg_channels].describe())

In [ ]:
raw.plot(n_channels=4, duration=5, scalings='auto', title='Butterfly EEG View');

In [ ]:
raw.plot(start = 15, duration = 5, scalings = "auto");

In [ ]:
# Pick a timepoint in seconds (e.g., 1s)
raw.plot_sensors(show_names=True, kind='topomap', title='Electrode Positions')
raw.plot_psd(area_mode='range', average=True)  # Frequency domain


In [ ]:
# Step 7: Add accelerometer & gyroscope as annotations (optional advanced)
# ------------------------------------------------
# Create MNE annotations for motion (e.g., high acc)
df['Acc_norm'] = np.sqrt(df['AccX']**2 + df['AccY']**2 + df['AccZ']**2)
acc_threshold = df['Acc_norm'].quantile(0.95)
motion_indices = df.index[df['Acc_norm'] > acc_threshold].tolist()

onset_times = [i / 250 for i in motion_indices]
durations = [0.1] * len(onset_times)  # 100ms segments
descriptions = ['motion'] * len(onset_times)
annotations = mne.Annotations(onset=onset_times, duration=durations, description=descriptions)
raw.set_annotations(annotations)

# Plot again with motion highlighted
raw.plot(n_channels=4, duration=5, scalings='auto', title='With Motion Annotations')
